# NVIDIA CUDA Feature Notebook for PyTorch / Python

This notebook checks which CUDA/NVIDIA features work on your current GPU and marks the minimum compute capability (`sm_xx`) usually required.

Your reported GPU: **NVIDIA GeForce GTX 950M**, compute capability **sm_50**.

General rule: sections marked `sm_50` should work on your GPU. Sections requiring `sm_70`, `sm_75`, `sm_80`, or higher need a newer GPU.

The notebook uses mostly **PyTorch**, plus optional NVIDIA/Python ecosystem packages when installed:

- `torch` for CUDA tensors, kernels, streams, events, matmul, attention, and graphs
- `pynvml` / `nvidia-ml-py` for GPU monitoring
- `numba.cuda` for simple custom CUDA kernels
- `cupy` for CUDA array programming, if available

Advanced features are at the bottom.

## 0. Environment and GPU capability check

**Minimum SM:** any CUDA-capable GPU.

This cell prints the PyTorch version, CUDA runtime version used by PyTorch, whether CUDA is available, GPU name, compute capability, and the architecture list included in your PyTorch build.

In [4]:
import sys
import platform
import torch

print("Python", sys.version)
print("Platform", platform.platform())
print("torch", torch.__version__)
print("CUDA built into PyTorch", torch.version.cuda)
print("CUDA available", torch.cuda.is_available())

assert torch.cuda.is_available(), "CUDA is not available in this kernel"

device = torch.device("cuda:0")
props = torch.cuda.get_device_properties(0)
capability = torch.cuda.get_device_capability(0)
sm = capability[0] * 10 + capability[1]

print("device", torch.cuda.get_device_name(0))
print("capability", capability, f"sm_{sm}")
print("arch list", torch.cuda.get_arch_list())
print("total VRAM GB", round(props.total_memory / 1024**3, 2))
print("multiprocessors", props.multi_processor_count)
print("shared memory per block KB", props.shared_memory_per_block // 1024)

def supports(required_sm: int) -> bool:
    return sm >= required_sm

def report_feature(name: str, required_sm: int):
    status = "YES" if supports(required_sm) else "NO"
    print(f"{name:45s} requires sm_{required_sm:<3d} | this GPU sm_{sm:<3d} | supported: {status}")

Python 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Platform Windows-10-10.0.19045-SP0
torch 2.12.1+cu126
CUDA built into PyTorch 12.6
CUDA available True
device NVIDIA GeForce GTX 950M
capability (5, 0) sm_50
arch list ['sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
total VRAM GB 4.0
multiprocessors 5
shared memory per block KB 48


## 1. Basic CUDA tensors and device transfer

**Minimum SM:** `sm_50` works.

This is the core PyTorch CUDA workflow: move tensors to GPU, run operations, and bring results back to CPU.

In [5]:
report_feature("Basic CUDA tensors", 50)

x_cpu = torch.randn(4, 4)
x_gpu = x_cpu.to(device)
y_gpu = x_gpu * 2 + 1

print("x device:", x_gpu.device)
print("y device:", y_gpu.device)
print("result on CPU:", y_gpu.cpu())

Basic CUDA tensors                            requires sm_50  | this GPU sm_50  | supported: YES
x device: cuda:0
y device: cuda:0
result on CPU: tensor([[-0.7027,  2.7250,  2.6002,  3.4579],
        [-2.1577,  1.0606, -1.0458, -0.7332],
        [ 0.9477, -1.8061,  1.2514,  1.8177],
        [-0.7613, -3.7055,  1.5546,  2.6221]])


## 2. FP32 matrix multiplication

**Minimum SM:** `sm_50` works.

Your GTX 950M can run FP32 matrix multiplication through CUDA/cuBLAS. This is the safest numeric mode for your GPU.

In [6]:
report_feature("FP32 matmul / cuBLAS", 50)

a = torch.randn(1024, 1024, device=device, dtype=torch.float32)
b = torch.randn(1024, 1024, device=device, dtype=torch.float32)

# Warmup
c = a @ b
torch.cuda.synchronize()

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()
c = a @ b
end.record()
torch.cuda.synchronize()

print("shape", tuple(c.shape), "dtype", c.dtype)
print("elapsed ms", start.elapsed_time(end))

FP32 matmul / cuBLAS                          requires sm_50  | this GPU sm_50  | supported: YES
shape (1024, 1024) dtype torch.float32
elapsed ms 2.3393280506134033


## 3. FP64 / double precision check

**Minimum SM:** `sm_50` works, but consumer GPUs usually have very slow FP64.

This is useful for scientific correctness tests, but it is normally much slower than FP32 on mobile/consumer NVIDIA GPUs.

In [7]:
report_feature("FP64 tensors", 50)

a64 = torch.randn(512, 512, device=device, dtype=torch.float64)
b64 = torch.randn(512, 512, device=device, dtype=torch.float64)
c64 = a64 @ b64
print("dtype", c64.dtype, "mean", c64.mean().item())

FP64 tensors                                  requires sm_50  | this GPU sm_50  | supported: YES
dtype torch.float64 mean 0.05227907243269908


## 4. FP16 tensors: supported but usually not accelerated on sm_50

**Minimum SM:** `sm_50` can store and operate on FP16 tensors in many cases.

**Important limitation:** GTX 950M does **not** have Tensor Cores. FP16 is not a good performance path on this GPU and can be slower or less stable than FP32.

In [8]:
report_feature("FP16 tensors without Tensor Cores", 50)

try:
    a16 = torch.randn(512, 512, device=device, dtype=torch.float16)
    b16 = torch.randn(512, 512, device=device, dtype=torch.float16)
    c16 = a16 @ b16
    torch.cuda.synchronize()
    print("FP16 matmul worked. dtype:", c16.dtype, "mean:", float(c16.float().mean().cpu()))
except Exception as e:
    print("FP16 operation failed on this setup:", repr(e))

FP16 tensors without Tensor Cores             requires sm_50  | this GPU sm_50  | supported: YES
FP16 matmul worked. dtype: torch.float16 mean: 0.008362801745533943


## 5. BF16 check

**Minimum SM for useful BF16 acceleration:** usually `sm_80` / Ampere or newer.

Your GTX 950M / `sm_50` does not have BF16 hardware acceleration. PyTorch may reject BF16 CUDA ops or fall back depending on the operation/build.

In [9]:
report_feature("BF16 hardware acceleration", 80)
print("torch.cuda.is_bf16_supported():", torch.cuda.is_bf16_supported())

try:
    abf = torch.randn(256, 256, device=device, dtype=torch.bfloat16)
    bbf = torch.randn(256, 256, device=device, dtype=torch.bfloat16)
    cbf = abf @ bbf
    torch.cuda.synchronize()
    print("BF16 matmul result dtype:", cbf.dtype)
except Exception as e:
    print("BF16 operation not available or not useful here:", repr(e))

BF16 hardware acceleration                    requires sm_80  | this GPU sm_50  | supported: NO
torch.cuda.is_bf16_supported(): True
BF16 matmul result dtype: torch.bfloat16


## 6. cuDNN convolution with PyTorch

**Minimum SM:** `sm_50` works for many standard convolutions.

This tests a small CNN-style convolution. This is a suitable class of workload for your GPU, although modern GPUs are much faster.

In [10]:
report_feature("cuDNN / convolution", 50)

conv = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1).to(device)
x = torch.randn(8, 3, 128, 128, device=device)

y = conv(x)
torch.cuda.synchronize()
print("output shape", tuple(y.shape), "dtype", y.dtype)
print("cuDNN enabled", torch.backends.cudnn.enabled)
print("cuDNN version", torch.backends.cudnn.version())

cuDNN / convolution                           requires sm_50  | this GPU sm_50  | supported: YES
output shape (8, 16, 128, 128) dtype torch.float32
cuDNN enabled True
cuDNN version 91002


## 7. CUDA streams and asynchronous execution

**Minimum SM:** `sm_50` works.

Streams allow overlapping and ordering GPU work. This is a core CUDA feature and useful for advanced data pipelines.

In [11]:
report_feature("CUDA streams", 50)

s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

x1 = torch.randn(2048, 2048, device=device)
x2 = torch.randn(2048, 2048, device=device)

with torch.cuda.stream(s1):
    y1 = x1 * 2.0
with torch.cuda.stream(s2):
    y2 = x2 + 3.0

torch.cuda.synchronize()
print("stream results", y1.mean().item(), y2.mean().item())

CUDA streams                                  requires sm_50  | this GPU sm_50  | supported: YES
stream results -0.0003178835613653064 3.0


## 8. CUDA events for timing

**Minimum SM:** `sm_50` works.

CUDA events are the recommended way to time GPU operations because CUDA calls are asynchronous from Python.

In [12]:
report_feature("CUDA events", 50)

x = torch.randn(4096, 4096, device=device)
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
y = torch.relu(x)
end.record()
torch.cuda.synchronize()

print("ReLU elapsed ms", start.elapsed_time(end))

CUDA events                                   requires sm_50  | this GPU sm_50  | supported: YES
ReLU elapsed ms 33.89219284057617


## 9. GPU memory management

**Minimum SM:** `sm_50` works.

This shows allocated and reserved memory from PyTorch's CUDA caching allocator.

In [13]:
report_feature("PyTorch CUDA memory management", 50)

torch.cuda.empty_cache()
print("allocated MB before", round(torch.cuda.memory_allocated() / 1024**2, 2))
print("reserved MB before", round(torch.cuda.memory_reserved() / 1024**2, 2))

tmp = torch.randn(1024, 1024, device=device)
print("allocated MB after", round(torch.cuda.memory_allocated() / 1024**2, 2))
print("reserved MB after", round(torch.cuda.memory_reserved() / 1024**2, 2))

del tmp
torch.cuda.empty_cache()
print("allocated MB final", round(torch.cuda.memory_allocated() / 1024**2, 2))
print("reserved MB final", round(torch.cuda.memory_reserved() / 1024**2, 2))

PyTorch CUDA memory management                requires sm_50  | this GPU sm_50  | supported: YES
allocated MB before 220.0
reserved MB before 234.0
allocated MB after 224.0
reserved MB after 234.0
allocated MB final 220.0
reserved MB final 234.0


## 10. NVML monitoring from Python

**Minimum SM:** independent of SM; requires NVIDIA driver and `nvidia-ml-py` / `pynvml` package.

Install if needed:

```bash
pip install nvidia-ml-py
```

This reads GPU utilization, temperature, and memory through NVIDIA Management Library.

In [14]:
try:
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    name = pynvml.nvmlDeviceGetName(handle)
    util = pynvml.nvmlDeviceGetUtilizationRates(handle)
    mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
    temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
    print("NVML device", name)
    print("GPU utilization %", util.gpu)
    print("Memory utilization %", util.memory)
    print("Memory used MB", round(mem.used / 1024**2, 2), "/", round(mem.total / 1024**2, 2))
    print("Temperature C", temp)
    pynvml.nvmlShutdown()
except Exception as e:
    print("NVML not available. Install with: pip install nvidia-ml-py")
    print("Error:", repr(e))

NVML not available. Install with: pip install nvidia-ml-py
Error: ModuleNotFoundError("No module named 'pynvml'")


## 11. Custom CUDA kernel with Numba

**Minimum SM:** `sm_50` works if Numba supports your installed CUDA driver/toolchain.

Install if needed:

```bash
pip install numba
```

This is useful for learning CUDA programming from Python.

In [15]:
try:
    import numpy as np
    from numba import cuda

    @cuda.jit
    def add_kernel(a, b, out):
        i = cuda.grid(1)
        if i < out.size:
            out[i] = a[i] + b[i]

    n = 1_000_000
    a = np.ones(n, dtype=np.float32)
    b = np.ones(n, dtype=np.float32) * 2
    out = np.empty_like(a)

    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_out = cuda.to_device(out)

    threads = 256
    blocks = (n + threads - 1) // threads
    add_kernel[blocks, threads](d_a, d_b, d_out)
    d_out.copy_to_host(out)

    print("Numba CUDA worked:", out[:5])
except Exception as e:
    print("Numba CUDA not available in this environment:", repr(e))

Numba CUDA not available in this environment: ModuleNotFoundError("No module named 'numba'")


## 12. CuPy CUDA arrays

**Minimum SM:** `sm_50` usually works if you install a CuPy package compatible with your CUDA runtime.

Install example for CUDA 12.x:

```bash
pip install cupy-cuda12x
```

CuPy gives a NumPy-like API on CUDA arrays.

In [16]:
try:
    import cupy as cp
    x = cp.random.randn(1024, 1024, dtype=cp.float32)
    y = cp.maximum(x, 0)
    cp.cuda.Stream.null.synchronize()
    print("CuPy worked. shape:", y.shape, "mean:", float(y.mean()))
except Exception as e:
    print("CuPy not available in this environment:", repr(e))

CuPy not available in this environment: ModuleNotFoundError("No module named 'cupy'")


# Advanced features

The remaining sections are more likely to require newer GPUs, newer CUDA libraries, or specific PyTorch builds.

## 13. Tensor Cores

**Minimum SM:** `sm_70` for Volta Tensor Cores; `sm_75`/`sm_80`+ for more modern Tensor Core paths.

Your GTX 950M / `sm_50` does **not** have Tensor Cores. FP16/BF16/TF32 acceleration is therefore unavailable.

In [17]:
report_feature("Tensor Cores", 70)

if supports(70):
    a = torch.randn(2048, 2048, device=device, dtype=torch.float16)
    b = torch.randn(2048, 2048, device=device, dtype=torch.float16)
    c = a @ b
    torch.cuda.synchronize()
    print("Tensor-core-capable GPU path may be used. result dtype:", c.dtype)
else:
    print("No Tensor Cores on this GPU. Use an RTX/Tesla/A100/H100-class GPU to test this properly.")

Tensor Cores                                  requires sm_70  | this GPU sm_50  | supported: NO
No Tensor Cores on this GPU. Use an RTX/Tesla/A100/H100-class GPU to test this properly.


## 14. TF32 matrix multiplication

**Minimum SM:** `sm_80` / Ampere or newer.

TF32 accelerates FP32-like matrix multiplication on Ampere+ Tensor Cores. It is unavailable on GTX 950M.

In [18]:
report_feature("TF32 matmul", 80)
print("matmul allow_tf32:", torch.backends.cuda.matmul.allow_tf32)
print("cuDNN allow_tf32:", torch.backends.cudnn.allow_tf32)

if supports(80):
    torch.backends.cuda.matmul.allow_tf32 = True
    a = torch.randn(4096, 4096, device=device)
    b = torch.randn(4096, 4096, device=device)
    c = a @ b
    torch.cuda.synchronize()
    print("TF32 may be used internally for FP32 matmul on Ampere+.")
else:
    print("TF32 requires Ampere or newer, so this GPU cannot use it.")

TF32 matmul                                   requires sm_80  | this GPU sm_50  | supported: NO
matmul allow_tf32: False
cuDNN allow_tf32: True
TF32 requires Ampere or newer, so this GPU cannot use it.


## 15. PyTorch scaled dot product attention backend check

**Minimum SM:** math backend works on `sm_50`; efficient/flash kernels usually require newer GPUs.

This cell runs PyTorch SDPA. On your GPU, it should use a fallback/math-style implementation, not FlashAttention.

In [19]:
import torch.nn.functional as F

report_feature("SDPA math fallback", 50)
report_feature("Flash-style attention kernels", 75)

batch, heads, seq, dim = 1, 2, 64, 32
q = torch.randn(batch, heads, seq, dim, device=device)
k = torch.randn(batch, heads, seq, dim, device=device)
v = torch.randn(batch, heads, seq, dim, device=device)

out = F.scaled_dot_product_attention(q, k, v)
torch.cuda.synchronize()
print("SDPA output shape", tuple(out.shape), "dtype", out.dtype)
print("Expected on sm_50: math/fallback attention, not FlashAttention.")

SDPA math fallback                            requires sm_50  | this GPU sm_50  | supported: YES
Flash-style attention kernels                 requires sm_75  | this GPU sm_50  | supported: NO
SDPA output shape (1, 2, 64, 32) dtype torch.float32
Expected on sm_50: math/fallback attention, not FlashAttention.


## 16. `torch.compile` / Inductor

**Minimum SM:** can work on `sm_50`, but many generated/Triton kernels and optimizations are targeted at newer GPUs.

This is worth trying, but do not expect large speedups on GTX 950M. Some backends may fail depending on your environment.

In [20]:
report_feature("torch.compile basic", 50)

if hasattr(torch, "compile"):
    def f(x, w):
        return torch.relu(x @ w)

    x = torch.randn(512, 512, device=device)
    w = torch.randn(512, 512, device=device)

    try:
        f_compiled = torch.compile(f)
        y = f_compiled(x, w)
        torch.cuda.synchronize()
        print("torch.compile worked. output mean:", y.mean().item())
    except Exception as e:
        print("torch.compile failed or backend unsupported here:", repr(e))
else:
    print("torch.compile is not available in this PyTorch version.")

torch.compile basic                           requires sm_50  | this GPU sm_50  | supported: YES


W0705 17:50:16.309000 21908 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
W0705 17:50:58.513000 21908 torch\_inductor\utils.py:1717] [0/0] Not enough SMs to use max_autotune_gemm mode


torch.compile failed or backend unsupported here: GPUTooOldForTriton('Found NVIDIA GeForce GTX 950M which is too old to be supported by the triton GPU compiler, which is used as the backend. Triton only supports devices of CUDA Capability >= 7.0, but your device is of CUDA capability 5.0\n\nSet TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you\'re reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"\n')


## 17. CUDA Graphs

**Minimum SM:** `sm_50` can support CUDA Graph capture if the driver/runtime and operations are compatible.

CUDA Graphs reduce CPU launch overhead by capturing and replaying a fixed sequence of GPU operations. Shapes and memory addresses must stay stable.

In [21]:
report_feature("CUDA Graphs", 50)

try:
    static_x = torch.randn(1024, 1024, device=device)
    static_w = torch.randn(1024, 1024, device=device)
    static_y = torch.empty_like(static_x)

    # Warmup on side stream
    s = torch.cuda.Stream()
    s.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(s):
        for _ in range(3):
            static_y.copy_(torch.relu(static_x @ static_w))
    torch.cuda.current_stream().wait_stream(s)

    g = torch.cuda.CUDAGraph()
    with torch.cuda.graph(g):
        static_y.copy_(torch.relu(static_x @ static_w))

    g.replay()
    torch.cuda.synchronize()
    print("CUDA Graph replay worked. mean:", static_y.mean().item())
except Exception as e:
    print("CUDA Graph example failed on this setup:", repr(e))

CUDA Graphs                                   requires sm_50  | this GPU sm_50  | supported: YES
CUDA Graph replay worked. mean: 12.758988380432129


## 18. Triton custom kernel

**Minimum SM:** often `sm_70`+ in practice, depending on Triton version and kernel.

Many modern PyTorch/Triton optimization paths do not target `sm_50`. Try this only on a newer GPU if it fails here.

In [22]:
report_feature("Triton practical support", 70)

try:
    import triton
    import triton.language as tl

    @triton.jit
    def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
        pid = tl.program_id(axis=0)
        offsets = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offsets < n_elements
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        tl.store(out_ptr + offsets, x + y, mask=mask)

    n = 1024
    x = torch.ones(n, device=device)
    y = torch.ones(n, device=device) * 2
    out = torch.empty_like(x)

    grid = lambda meta: (triton.cdiv(n, meta["BLOCK"]),)
    add_kernel[grid](x, y, out, n, BLOCK=256)
    torch.cuda.synchronize()
    print("Triton worked:", out[:5])
except Exception as e:
    print("Triton unavailable or not supported on this GPU/environment:", repr(e))

Triton practical support                      requires sm_70  | this GPU sm_50  | supported: NO
Triton unavailable or not supported on this GPU/environment: ModuleNotFoundError("No module named 'triton'")


## 19. Quantized LLM kernels: bitsandbytes / GPTQ / AWQ

**Minimum SM:** varies by package and kernel; many practical fast kernels expect `sm_60`, `sm_70`, `sm_75`, or newer.

On GTX 950M / `sm_50`, many quantized LLM packages may not install, may fall back, or may run without the expected speedups.

This cell only checks whether `bitsandbytes` imports and reports basic information.

In [23]:
report_feature("Modern quantized LLM kernels", 70)

try:
    import bitsandbytes as bnb
    print("bitsandbytes imported:", bnb.__version__ if hasattr(bnb, "__version__") else bnb)
    print("Import success does not guarantee fast kernels on sm_50.")
except Exception as e:
    print("bitsandbytes not available or unsupported here:", repr(e))

Modern quantized LLM kernels                  requires sm_70  | this GPU sm_50  | supported: NO
bitsandbytes not available or unsupported here: ModuleNotFoundError("No module named 'bitsandbytes'")


## 20. FP8 / Transformer Engine

**Minimum SM:** usually `sm_90` / Hopper for NVIDIA Transformer Engine FP8 workflows.

Not available on GTX 950M. This is for H100/H200/Blackwell-class experimentation.

In [24]:
report_feature("FP8 / Transformer Engine", 90)

try:
    import transformer_engine
    print("Transformer Engine imported:", transformer_engine)
    if supports(90):
        print("This GPU may support FP8 workflows depending on package/version.")
    else:
        print("Package import does not matter: this GPU lacks FP8 Tensor Core support.")
except Exception as e:
    print("Transformer Engine not installed or unsupported:", repr(e))

FP8 / Transformer Engine                      requires sm_90  | this GPU sm_50  | supported: NO
Transformer Engine not installed or unsupported: ModuleNotFoundError("No module named 'transformer_engine'")


## 21. Summary table for your GPU

This table gives a quick pass/fail view based on compute capability only. Package, driver, OS, and PyTorch build may impose additional constraints.

In [25]:
features = [
    ("Basic CUDA tensors", 50),
    ("FP32 matmul / cuBLAS", 50),
    ("FP64 tensors", 50),
    ("FP16 tensors without Tensor Cores", 50),
    ("cuDNN convolution", 50),
    ("CUDA streams/events", 50),
    ("CUDA Graphs", 50),
    ("Numba CUDA basic kernels", 50),
    ("CuPy basic arrays", 50),
    ("Tensor Cores", 70),
    ("Flash-style attention kernels", 75),
    ("BF16 acceleration", 80),
    ("TF32", 80),
    ("FP8 / Transformer Engine", 90),
]

print(f"Current GPU: {torch.cuda.get_device_name(0)} / sm_{sm}")
print("-" * 76)
for name, req in features:
    status = "works / worth trying" if supports(req) else "needs newer GPU"
    print(f"{name:38s} | min sm_{req:<3d} | {status}")

Current GPU: NVIDIA GeForce GTX 950M / sm_50
----------------------------------------------------------------------------
Basic CUDA tensors                     | min sm_50  | works / worth trying
FP32 matmul / cuBLAS                   | min sm_50  | works / worth trying
FP64 tensors                           | min sm_50  | works / worth trying
FP16 tensors without Tensor Cores      | min sm_50  | works / worth trying
cuDNN convolution                      | min sm_50  | works / worth trying
CUDA streams/events                    | min sm_50  | works / worth trying
CUDA Graphs                            | min sm_50  | works / worth trying
Numba CUDA basic kernels               | min sm_50  | works / worth trying
CuPy basic arrays                      | min sm_50  | works / worth trying
Tensor Cores                           | min sm_70  | needs newer GPU
Flash-style attention kernels          | min sm_75  | needs newer GPU
BF16 acceleration                      | min sm_80  | needs new